In [2]:
# ==========================================================
# Project Setup
# ==========================================================

import sys
from pathlib import Path


# 获取项目根目录
PROJECT_ROOT = Path("..").resolve()


# 加入 Python 模块搜索路径
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


print("Project root:", PROJECT_ROOT)

Project root: D:\desktop\work-projects\breast-ultrasound-segmentation


### 测试ConvBlock模块---养成良好的习惯，写一个模块测试一个模块

In [2]:
#将项目根目录加入python模块搜索路径
import sys
from pathlib import Path

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
import torch
from src.model import ConvBlock

block = ConvBlock(
    in_channels=1,
    out_channels=64
)
#制造一批假的超声数据
x = torch.randn(
    8,
    1,
    256,
    256
)
print("输入shape:",x.shape)
y = block(x)
print("输出shape:",y.shape)
"""
模型已经把原始灰度图变成了64个特征通道
"""

输入shape: torch.Size([8, 1, 256, 256])
输出shape: torch.Size([8, 64, 256, 256])


'\n模型已经把原始灰度图变成了64个特征通道\n'

#### 测试encoder

In [3]:
import torch
from src.model import UNet

model = UNet()
x = torch.randn(
    8,
    1,
    256,
    256
)

e1, e2, e3, e4 = model(x)

print("Input:", x.shape)

print("Encoder 1:", e1.shape)
print("Encoder 2:", e2.shape)
print("Encoder 3:", e3.shape)
print("Encoder 4:", e4.shape)

Input: torch.Size([8, 1, 256, 256])
Encoder 1: torch.Size([8, 64, 256, 256])
Encoder 2: torch.Size([8, 128, 128, 128])
Encoder 3: torch.Size([8, 256, 64, 64])
Encoder 4: torch.Size([8, 512, 32, 32])


##### Input
[8,1,256,256]

        ↓ ConvBlock

e1
[8,64,256,256]

        ↓ MaxPool

[8,64,128,128]

        ↓ ConvBlock

e2
[8,128,128,128]

        ↓ MaxPool

[8,128,64,64]

        ↓ ConvBlock

e3
[8,256,64,64]

        ↓ MaxPool

[8,256,32,32]

        ↓ ConvBlock

e4
[8,512,32,32]

In [4]:
import torch
from src.model import UNet

model = UNet()  #少了一个括号，报错，UNet是一个类，表示模型模板；但是UNet()表示真正创建出来的模型对象
#制造假的输入
x = torch.randn(
    8,
    1,
    256,
    256
)
output = model(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)


Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 1, 256, 256])


In [ ]:
#====================================
#统计模型参数量
#训练的本质：一开始卷积核权重随机→计算loss→反向传播→不断修改这些参数→模型逐渐学会分割病灶
#====================================
total_params = sum(
    p.numel()
    for p in model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad  #这个参数需不需要通过反向传播被更新？如果不需要训练就冻结这个参数不训练
)
print("总参数量：",total_params)
print("可训练参数量：",trainable_params)

总参数量： 31036481
可训练参数量： 31036481


In [6]:
print(model)

UNet(
  (enc1): ConvBlock(
    (block): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): ConvBlock(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    